In [1]:
%reload_ext autoreload
%autoreload 2

# Imports

In [2]:
from kret_notebook import *  # NOTE import first
from kret_lgbm._core.lgbm_nb_imports import *
from kret_lightning._core.lightning_nb_imports import *
from kret_matplotlib._core.mpl_nb_imports import *
from kret_np_pd._core.np_pd_nb_imports import *
from kret_optuna._core.optuna_nb_imports import *
from kret_polars._core.polars_nb_imports import *
from kret_rosetta._core.rosetta_nb_imports import *
from kret_sklearn._core.sklearn_nb_imports import *
from kret_torch_utils._core.torch_nb_imports import *
from kret_tqdm._core.tqdm_nb_imports import *
from kret_type_hints._core.types_nb_imports import *
from kret_utils._core.utils_nb_imports import *

# from kret_wandb._core.wandb_nb_imports import *  # NOTE this is slow to import

Loaded environment variables from /Users/Akseldkw/coding/Columbia/NBA-Timeout-Impact/.env
[kret_lgbm._core.lgbm_nb_imports] Imported kret_lgbm._core.lgbm_nb_imports in 2.0302 seconds
[kret_lightning._core.lightning_nb_imports] Imported kret_lightning._core.lightning_nb_imports in 4.0273 seconds
[kret_matplotlib._core.mpl_nb_imports] Imported kret_matplotlib._core.mpl_nb_imports in 0.3785 seconds
[kret_np_pd._core.np_pd_nb_imports] Imported kret_np_pd._core.np_pd_nb_imports in 0.0006 seconds
[kret_optuna._core.optuna_nb_imports] Imported kret_optuna._core.optuna_nb_imports in 0.0009 seconds
[kret_polars._core.polars_nb_imports] Imported kret_polars._core.polars_nb_imports in 0.0000 seconds
[kret_rosetta._core.rosetta_nb_imports] Imported kret_rosetta._core.rosetta_nb_imports in 0.0000 seconds
[kret_sklearn._core.sklearn_nb_imports] Imported kret_sklearn._core.sklearn_nb_imports in 0.0858 seconds
[kret_torch_utils._core.torch_nb_imports] Imported kret_torch_utils._core.torch_nb_imports i

In [3]:
from nba_timeout_impact.nb_imports import *

# Load Data

In [15]:
import time
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util import Retry
from nba_api.stats.endpoints import playbyplayv3


def get_pbp_safely(game_id):
    # 1. Advanced Akamai CDN Bypass Header configuration
    headers = {
        "Host": "stats.nba.com",
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://www.nba.com/",
        "Origin": "https://www.nba.com",
        "Connection": "keep-alive",
        "x-nba-stats-origin": "stats",
        "x-nba-stats-token": "true",
    }

    # 2. Establish explicit retry limits to prevent infinite 30-redirect loops
    # This forces the client to back off rather than blindly following redirects
    pbp_endpoint = playbyplayv3.PlayByPlayV3(game_id=game_id, headers=headers, timeout=15)

    # Extract structural list entries
    frames = pbp_endpoint.get_data_frames()
    return frames[0]


# Guaranteed 2024-2025 historic baseline game ID string
target_game = "0022400001"

try:
    print(f"Attempting secure connection to extract game: {target_game}...")
    df_raw = get_pbp_safely(target_game)

    # Filter for Timeouts (Event type 9)
    df_timeouts = df_raw[df_raw["EVENTMSGTYPE"] == 9].copy()

    print("\n--- Timeout Data Retrieved Successfully ---")
    print(df_timeouts[["PERIOD", "PCTIMESTRING", "HOMEDESCRIPTION", "VISITORDESCRIPTION"]].to_string())

except Exception as e:
    print(f"\nAPI Access Denied: {e}")
    print("Alternative Action: The CDN firewall is aggressively blocking your local IP.")
    print("Switch to the pre-scraped SQLite DB mirror on Kaggle to bypass live scraping entirely.")

Attempting secure connection to extract game: 0022400001...

API Access Denied: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=15)
Alternative Action: The CDN firewall is aggressively blocking your local IP.
Switch to the pre-scraped SQLite DB mirror on Kaggle to bypass live scraping entirely.


In [14]:
from nba_timeout_impact.datasets.memo_nbastatsv3 import NBAMemoDF
from nba_timeout_impact.analyses.tv_timeout_validation import *

memo = NBAMemoDF.load_all()

In [ ]:
compare_configs(memo, [POST_2017, *PRE_2017_CANDIDATES])

,config,n_gt,n_pred,TP,FP,FN,precision,recall,f1
0,"post-2017 (7:00, 3:00) x Q1-Q4",42198,64738,2373,62365,39825,0.0367,0.0562,0.0444
1,"pre-2017 (9:00, 3:00) x Q1-Q4",42198,78158,23867,54291,18331,0.3054,0.5656,0.3966
2,"pre-2017 (9:00, 6:00, 3:00) x Q1-Q4",42198,91487,30334,61153,11864,0.3316,0.7188,0.4538
3,"pre-2017 (9:00, 6:00, 3:00) x Q2/Q4 + (6:00) x...",41418,44883,30270,14613,11148,0.6744,0.7308,0.7015


# Implementation

# Sandbox